# Phase 2 — Collecte & Audit des Données
## Dataset : Hillstrom E-Mail Analytics
**Objectif :** Cartographier les sources, auditer la qualité des données, 
documenter le Data Dictionary et merger les deux sources.

### Plan de cette phase :
1. Chargement et première inspection du dataset
2. Audit de qualité complet (manquants, doublons, cohérence)
3. Data Dictionary
4. Chargement de la deuxième source (données démographiques)
5. Merge et validation finale
6. Conclusion de l'audit

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)
print("✅ Librairies chargées")

✅ Librairies chargées


In [7]:
df = pd.read_csv('data/hillstrom.csv')
print(f"✅ Dataset chargé")
print(f"📊 Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"\n📋 Colonnes : {list(df.columns)}")
df.head()

✅ Dataset chargé
📊 Dimensions : 64,000 lignes × 12 colonnes

📋 Colonnes : ['recency', 'history_segment', 'history', 'mens', 'womens', 'zip_code', 'newbie', 'channel', 'segment', 'visit', 'conversion', 'spend']


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.440,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.000
1,6,3) $200 - $350,329.080,1,1,Rural,1,Web,No E-Mail,0,0,0.000
2,7,2) $100 - $200,180.650,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.000
3,9,5) $500 - $750,675.830,1,0,Rural,1,Web,Mens E-Mail,0,0,0.000
4,2,1) $0 - $100,45.340,1,0,Urban,0,Web,Womens E-Mail,0,0,0.000


In [8]:
print("=== AUDIT DE QUALITE ===\n")
print("1. Types des colonnes :")
print(df.dtypes)
print(f"\n2. Valeurs manquantes :")
print(df.isnull().sum())
print(f"\n3. Doublons : {df.duplicated().sum()}")
print(f"\n4. Statistiques descriptives :")
df.describe()

=== AUDIT DE QUALITE ===

1. Types des colonnes :
recency              int64
history_segment        str
history            float64
mens                 int64
womens               int64
zip_code               str
newbie               int64
channel                str
segment                str
visit                int64
conversion           int64
spend              float64
dtype: object

2. Valeurs manquantes :
recency            0
history_segment    0
history            0
mens               0
womens             0
zip_code           0
newbie             0
channel            0
segment            0
visit              0
conversion         0
spend              0
dtype: int64

3. Doublons : 6562

4. Statistiques descriptives :


,recency,history,mens,womens,newbie,visit,conversion,spend
count,64000.000,64000.000,64000.000,64000.000,64000.000,64000.000,64000.000,64000.000
mean,5.764,242.086,0.551,0.550,0.502,0.147,0.009,1.051
std,3.508,256.159,0.497,0.498,0.500,0.354,0.095,15.036
min,1.000,29.990,0.000,0.000,0.000,0.000,0.000,0.000
25%,2.000,64.660,0.000,0.000,0.000,0.000,0.000,0.000
50%,6.000,158.110,1.000,1.000,1.000,0.000,0.000,0.000
75%,9.000,325.657,1.000,1.000,1.000,0.000,0.000,0.000
max,12.000,3345.930,1.000,1.000,1.000,1.000,1.000,499.000


In [9]:
print("=== ANALYSE DES DOUBLONS ===")
print(f"Lignes totales : {len(df)}")
print(f"Doublons detectes : {df.duplicated().sum()}")
print(f"Pourcentage : {df.duplicated().sum()/len(df)*100:.1f}%")

print("\nExemple de lignes dupliquees :")
masque = df.duplicated(keep=False)
df[masque].sort_values(list(df.columns)).head(10)

=== ANALYSE DES DOUBLONS ===
Lignes totales : 64000
Doublons detectes : 6562
Pourcentage : 10.3%

Exemple de lignes dupliquees :


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
18473,1,1) $0 - $100,29.990,0,1,Rural,0,Phone,Mens E-Mail,0,0,0.000
38460,1,1) $0 - $100,29.990,0,1,Rural,0,Phone,Mens E-Mail,0,0,0.000
42755,1,1) $0 - $100,29.990,0,1,Rural,0,Phone,Mens E-Mail,0,0,0.000
47127,1,1) $0 - $100,29.990,0,1,Rural,0,Phone,Mens E-Mail,0,0,0.000
51350,1,1) $0 - $100,29.990,0,1,Rural,0,Phone,Mens E-Mail,0,0,0.000
3748,1,1) $0 - $100,29.990,0,1,Rural,0,Phone,Mens E-Mail,1,0,0.000
30423,1,1) $0 - $100,29.990,0,1,Rural,0,Phone,Mens E-Mail,1,0,0.000
50187,1,1) $0 - $100,29.990,0,1,Rural,0,Phone,Mens E-Mail,1,0,0.000
13562,1,1) $0 - $100,29.990,0,1,Rural,0,Phone,Womens E-Mail,0,0,0.000
21256,1,1) $0 - $100,29.990,0,1,Rural,0,Phone,Womens E-Mail,0,0,0.000


### Decision sur les doublons

6 562 lignes identifiées comme doublons (10.3% du dataset).

**Conclusion :** Ces doublons sont des coïncidences statistiques et non 
des erreurs de données. Le dataset Hillstrom ne contient pas de colonne 
customer_id — deux clients différents peuvent avoir exactement le même 
profil comportemental (même recency, même history, même segment).

**Decision :** Aucune suppression. Toutes les 64 000 lignes sont conservées.

In [10]:
print("=== DISTRIBUTION DES GROUPES EXPERIMENTAUX ===")
print(df['segment'].value_counts())
print(f"\nProportion :")
print(df['segment'].value_counts(normalize=True).round(3) * 100)

=== DISTRIBUTION DES GROUPES EXPERIMENTAUX ===
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64

Proportion :
segment
Womens E-Mail   33.400
Mens E-Mail     33.300
No E-Mail       33.300
Name: proportion, dtype: float64


### Validation de la randomisation

| Groupe | Clients | Proportion |
|---|---|---|
| Womens E-Mail | 21 387 | 33.4% |
| Mens E-Mail | 21 307 | 33.3% |
| No E-Mail (controle) | 21 306 | 33.3% |

**Conclusion :** Les 3 groupes sont parfaitement equilibres (ecart max 0.1%).
La randomisation est valide — les comparaisons entre groupes seront non biaisees.

In [11]:
print("=== TAUX DE CONVERSION PAR GROUPE ===")
conversion = df.groupby('segment').agg(
    clients=('conversion', 'count'),
    convertis=('conversion', 'sum'),
    taux_conversion=('conversion', 'mean'),
    spend_moyen=('spend', 'mean')
).round(4)

conversion['taux_conversion_pct'] = (conversion['taux_conversion'] * 100).round(2)
print(conversion)

=== TAUX DE CONVERSION PAR GROUPE ===
               clients  convertis  taux_conversion  spend_moyen  \
segment                                                           
Mens E-Mail      21307        267            0.013        1.423   
No E-Mail        21306        122            0.006        0.653   
Womens E-Mail    21387        189            0.009        1.077   

               taux_conversion_pct  
segment                             
Mens E-Mail                  1.250  
No E-Mail                    0.570  
Womens E-Mail                0.880  


### Premier insight — Effet causal de l'email

| Groupe | Taux conversion | Uplift vs controle |
|---|---|---|
| Mens E-Mail | 1.25% | +0.68% |
| Womens E-Mail | 0.88% | +0.31% |
| No E-Mail | 0.57% | reference |

**Conclusion :** L'email homme est 2x plus efficace que l'email femme.
Ces uplifts confirment l'existence de Persuadables dans la base client.
La modelisation va identifier ces clients precisement.

## 3. Data Dictionary

Description complète de chaque variable du dataset Hillstrom.

In [12]:
data_dictionary = {
    'recency': {
        'type': 'int64',
        'description': 'Nombre de mois depuis le dernier achat',
        'valeurs': '1 a 12',
        'source': 'Hillstrom'
    },
    'history_segment': {
        'type': 'str',
        'description': 'Tranche de depense historique du client',
        'valeurs': '1)$0-$100, 2)$100-$200, ..., 7)$1000+',
        'source': 'Hillstrom'
    },
    'history': {
        'type': 'float64',
        'description': 'Montant exact depense historiquement en dollars',
        'valeurs': 'valeur continue positive',
        'source': 'Hillstrom'
    },
    'mens': {
        'type': 'int64',
        'description': 'Client ayant achete dans le rayon homme',
        'valeurs': '0 = non, 1 = oui',
        'source': 'Hillstrom'
    },
    'womens': {
        'type': 'int64',
        'description': 'Client ayant achete dans le rayon femme',
        'valeurs': '0 = non, 1 = oui',
        'source': 'Hillstrom'
    },
    'zip_code': {
        'type': 'str',
        'description': 'Type de zone geographique du client',
        'valeurs': 'Urban, Suburban, Rural',
        'source': 'Hillstrom'
    },
    'newbie': {
        'type': 'int64',
        'description': 'Nouveau client (moins de 12 mois)',
        'valeurs': '0 = ancien, 1 = nouveau',
        'source': 'Hillstrom'
    },
    'channel': {
        'type': 'str',
        'description': 'Canal d achat habituel du client',
        'valeurs': 'Web, Phone, Multichannel',
        'source': 'Hillstrom'
    },
    'segment': {
        'type': 'str',
        'description': 'Groupe experimental assigne aleatoirement',
        'valeurs': 'Mens E-Mail, Womens E-Mail, No E-Mail',
        'source': 'Hillstrom'
    },
    'visit': {
        'type': 'int64',
        'description': 'A visite le site web apres la campagne',
        'valeurs': '0 = non, 1 = oui',
        'source': 'Hillstrom'
    },
    'conversion': {
        'type': 'int64',
        'description': 'A effectue un achat apres la campagne',
        'valeurs': '0 = non, 1 = oui',
        'source': 'Hillstrom'
    },
    'spend': {
        'type': 'float64',
        'description': 'Montant depense apres la campagne en dollars',
        'valeurs': 'valeur continue, 0 si pas d achat',
        'source': 'Hillstrom'
    }
}

dd = pd.DataFrame(data_dictionary).T
dd.index.name = 'variable'
print("=== DATA DICTIONARY ===")
dd

=== DATA DICTIONARY ===


,type,description,valeurs,source
variable,,,,
recency,int64,Nombre de mois depuis le dernier achat,1 a 12,Hillstrom
history_segment,str,Tranche de depense historique du client,"1)$0-$100, 2)$100-$200, ..., 7)$1000+",Hillstrom
history,float64,Montant exact depense historiquement en dollars,valeur continue positive,Hillstrom
mens,int64,Client ayant achete dans le rayon homme,"0 = non, 1 = oui",Hillstrom
womens,int64,Client ayant achete dans le rayon femme,"0 = non, 1 = oui",Hillstrom
zip_code,str,Type de zone geographique du client,"Urban, Suburban, Rural",Hillstrom
newbie,int64,Nouveau client (moins de 12 mois),"0 = ancien, 1 = nouveau",Hillstrom
channel,str,Canal d achat habituel du client,"Web, Phone, Multichannel",Hillstrom
segment,str,Groupe experimental assigne aleatoirement,"Mens E-Mail, Womens E-Mail, No E-Mail",Hillstrom


### Conclusion Data Dictionary

12 variables documentées :
- 5 variables numeriques continues : recency, history, spend, visit, conversion
- 4 variables binaires (0/1) : mens, womens, newbie, conversion, visit
- 3 variables categoriques : zip_code, channel, segment

Variable cible pour l'uplift : conversion (0/1)
Variable de traitement : segment (Mens E-Mail / Womens E-Mail / No E-Mail)

## Conclusion Phase 2 — Data Audit

### Bilan qualite des donnees
- 64 000 lignes, 12 colonnes
- 0 valeurs manquantes
- 6 562 doublons apparents — conserves (coïncidences statistiques, pas d erreurs)
- Randomisation validee : 3 groupes equilibres a 33%

### Premier insight
- Taux de conversion No E-Mail : 0.57% (reference)
- Taux de conversion Mens E-Mail : 1.25% (uplift +0.68%)
- Taux de conversion Womens E-Mail : 0.88% (uplift +0.31%)

### Decisions prises
- Aucune suppression de lignes
- Aucune imputation necessaire
- Deuxieme source demographique : sera integree en phase finale

### Donnees pretes pour la Phase 3 — EDA